In [1]:
from datasets import load_dataset
import numpy as np
from collections import Counter

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("lhoestq/conll2003")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [3]:
# ------------------------------
# Data preparation
# ------------------------------

def is_word_token(token):
    # Keep tokens that contain at least one letter or digit.
    return any(ch.isalnum() for ch in token)


def remove_punctuation_tokens(token_sequences):
    cleaned = []
    for sent in token_sequences:
        cleaned.append([tok for tok in sent if is_word_token(tok)])
    return cleaned


def build_vocab(token_sequences, max_vocab=None):
    counter = Counter(tok.lower() for sent in token_sequences for tok in sent if is_word_token(tok))
    vocab_items = [w for w, _ in counter.items()]
    vocab_items.sort(key=lambda w: counter[w], reverse=True)

    if max_vocab is not None:
        vocab_items = vocab_items[: max(0, max_vocab)]

    vocab = list(vocab_items)

    word_to_id = {w: i for i, w in enumerate(vocab)}
    id_to_word = {i: w for w, i in word_to_id.items()}
    return word_to_id, id_to_word, counter


def encode_sentences(token_sequences, word_to_id):
    encoded = []
    for sent in token_sequences:
        ids = []
        for tok in sent:
            if not is_word_token(tok):
                continue
            idx = word_to_id.get(tok.lower(), None)
            if idx is not None:
                ids.append(idx)
        if len(ids) > 1:
            encoded.append(ids)
    return encoded


def build_skipgram_pairs(encoded_sentences, window_size=2, max_pairs=None):
    centers, contexts = [], []
    for sent in encoded_sentences:
        n = len(sent)
        for i, center in enumerate(sent):
            left = max(0, i - window_size)
            right = min(n, i + window_size + 1)
            for j in range(left, right):
                if i == j:
                    continue
                centers.append(center)
                contexts.append(sent[j])
                if max_pairs is not None and len(centers) >= max_pairs:
                    print(f"Reached max_pairs={max_pairs}, stopping pair generation.")
                    return np.array(centers, dtype=np.int64), np.array(contexts, dtype=np.int64)
    return np.array(centers, dtype=np.int64), np.array(contexts, dtype=np.int64)

## Word2Vec (Skip-gram with Negative Sampling) Equations

Let:
- $v_c \in \mathbb{R}^d$: input embedding of center word $c$
- $u_o \in \mathbb{R}^d$: output embedding of true context word $o$
- $u_{n_k} \in \mathbb{R}^d$: output embedding of negative sample $n_k$
- $K$: number of negative samples
- $\sigma(x) = \frac{1}{1 + e^{-x}}$

### 1) Positive and negative scores
$$
s_{pos} = v_c^\top u_o
$$

$$
s_{neg,k} = v_c^\top u_{n_k}, \quad k=1,\dots,K
$$

### 2) Negative-sampling objective (single training pair)
$$
\mathcal{L}_{NS} = -\log\sigma\left(v_c^\top u_o\right) - \sum_{k=1}^{K} \log\sigma\left(-v_c^\top u_{n_k}\right)
$$

### 3) Mini-batch loss
For batch size $B$:
$$
\mathcal{L}_{batch} = \frac{1}{B} \sum_{i=1}^{B} \left[-\log\sigma\left(v_{c_i}^\top u_{o_i}\right) - \sum_{k=1}^{K}\log\sigma\left(-v_{c_i}^\top u_{n_{ik}}\right)\right]
$$

### 4) Useful derivatives used in backward pass
$$
\frac{d}{dx}\left[-\log\sigma(x)\right] = \sigma(x) - 1
$$

$$
\frac{d}{dx}\left[-\log\sigma(-x)\right] = \sigma(x)
$$

So for one sample:
$$
\frac{\partial \mathcal{L}}{\partial s_{pos}} = \sigma(s_{pos}) - 1
$$

$$
\frac{\partial \mathcal{L}}{\partial s_{neg,k}} = \sigma(s_{neg,k})
$$

### 5) Gradients w.r.t. embeddings (single sample)
Center embedding:
$$
\frac{\partial \mathcal{L}}{\partial v_c} = \left(\sigma(s_{pos}) - 1\right)u_o + \sum_{k=1}^{K}\sigma(s_{neg,k})u_{n_k}
$$

True context output embedding:
$$
\frac{\partial \mathcal{L}}{\partial u_o} = \left(\sigma(s_{pos}) - 1\right)v_c
$$

Negative output embeddings:
$$
\frac{\partial \mathcal{L}}{\partial u_{n_k}} = \sigma(s_{neg,k})v_c
$$

For mini-batch training, the same gradients are averaged by dividing by $B$.

### 6) Parameter update (SGD)
For learning rate $\eta$:
$$
\theta \leftarrow \theta - \eta \nabla_\theta \mathcal{L}_{batch}
$$
where $\theta$ includes input and output embedding matrices.

### 7) Similarity and analogy at inference
Cosine similarity between words $i,j$ (using input embeddings):
$$
\text{cos}(i,j)=\frac{v_i^\top v_j}{\|v_i\|\,\|v_j\|}
$$

Analogy target for "$a-b+c$":
$$
t = v_a - v_b + v_c
$$
Then retrieve nearest words to $t$ by cosine similarity.

In [4]:
# ------------------------------
# Scratch layers (manual backprop)
# ------------------------------

class Embedding:
    def __init__(self, vocab_size, emb_dim, seed=42):
        rng = np.random.default_rng(seed)
        self.W = rng.normal(0.0, 0.05, size=(vocab_size, emb_dim)).astype(np.float32)
        self.dW = np.zeros_like(self.W)

    def forward(self, indices):
        return self.W[indices]

    def zero_grad(self):
        self.dW.fill(0.0)

    def add_grad(self, indices, grad_values):
        np.add.at(self.dW, indices, grad_values)


class SGD:
    def __init__(self, parameters, lr=0.03):
        self.parameters = parameters
        self.lr = lr

    def step(self):
        for param, grad in self.parameters:
            param -= self.lr * grad


def sigmoid(x):
    x = np.clip(x, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-x))


class SkipGramNegativeSamplingScratch:
    def __init__(self, vocab_size, emb_dim=128, lr=0.03):
        self.input_embedding = Embedding(vocab_size=vocab_size, emb_dim=emb_dim, seed=42)
        self.output_embedding = Embedding(vocab_size=vocab_size, emb_dim=emb_dim, seed=123)
        self.optim = SGD(
            [
                (self.input_embedding.W, self.input_embedding.dW),
                (self.output_embedding.W, self.output_embedding.dW),
            ],
            lr=lr,
        )
        self.cache = None

    def forward(self, center_ids, context_ids, negative_ids):
        # Shapes: center_ids [B], context_ids [B], negative_ids [B, K]
        center_vecs = self.input_embedding.forward(center_ids)        # [B, D]
        pos_vecs = self.output_embedding.forward(context_ids)         # [B, D]
        neg_vecs = self.output_embedding.forward(negative_ids)        # [B, K, D]

        pos_scores = np.sum(center_vecs * pos_vecs, axis=1)          # [B]     vc . vo
        neg_scores = np.einsum("bd,bkd->bk", center_vecs, neg_vecs)  # [B, K]  vc . vn

        pos_sig = sigmoid(pos_scores)
        neg_sig = sigmoid(-neg_scores)

        eps = 1e-12
        loss = -np.mean(np.log(pos_sig + eps) + np.sum(np.log(neg_sig + eps), axis=1))

        self.cache = {
            "center_ids": center_ids,
            "context_ids": context_ids,
            "negative_ids": negative_ids,
            "center_vecs": center_vecs,
            "pos_vecs": pos_vecs,
            "neg_vecs": neg_vecs,
            "pos_scores": pos_scores,
            "neg_scores": neg_scores,
            "pos_sig": pos_sig,
        }
        return loss

    def backward(self):
        c = self.cache
        center_ids = c["center_ids"]
        context_ids = c["context_ids"]
        negative_ids = c["negative_ids"]
        center_vecs = c["center_vecs"]
        pos_vecs = c["pos_vecs"]
        neg_vecs = c["neg_vecs"]
        pos_sig = c["pos_sig"]
        neg_scores = c["neg_scores"]

        batch_size = center_vecs.shape[0]

        # d/dx[-log(sigmoid(x))] = sigmoid(x) - 1
        d_pos_scores = (pos_sig - 1.0) / batch_size                  # [B]

        # d/dx[-log(sigmoid(-x))] = sigmoid(x)
        d_neg_scores = sigmoid(neg_scores) / batch_size              # [B, K]

        grad_center = (
            d_pos_scores[:, None] * pos_vecs
            + np.einsum("bk,bkd->bd", d_neg_scores, neg_vecs)
        )                                                             # [B, D]

        grad_pos_out = d_pos_scores[:, None] * center_vecs           # [B, D]
        grad_neg_out = d_neg_scores[:, :, None] * center_vecs[:, None, :]  # [B, K, D]

        self.input_embedding.zero_grad()
        self.output_embedding.zero_grad()

        self.input_embedding.add_grad(center_ids, grad_center)
        self.output_embedding.add_grad(context_ids, grad_pos_out)
        self.output_embedding.add_grad(negative_ids, grad_neg_out)

    def step(self):
        self.optim.step()

In [5]:
# ------------------------------
# Train model (Negative Sampling)
# ------------------------------

def build_negative_sampling_distribution(encoded_sentences, vocab_size, power=0.75):
    counts = np.zeros(vocab_size, dtype=np.float64)
    for sent in encoded_sentences:
        counts += np.bincount(np.array(sent, dtype=np.int64), minlength=vocab_size)

    adjusted = np.power(counts, power)
    adjusted_sum = adjusted.sum()
    if adjusted_sum == 0:
        return np.ones(vocab_size, dtype=np.float64) / vocab_size
    return adjusted / adjusted_sum


def sample_negative_ids(batch_context_ids, num_negatives, neg_probs):
    vocab_size = len(neg_probs)
    neg_ids = np.random.choice(vocab_size, size=(len(batch_context_ids), num_negatives), p=neg_probs)

    # Avoid sampling the true context as a negative for each example.
    mask = neg_ids == batch_context_ids[:, None]
    while np.any(mask):
        neg_ids[mask] = np.random.choice(vocab_size, size=int(mask.sum()), p=neg_probs)
        mask = neg_ids == batch_context_ids[:, None]

    return neg_ids


train_tokens = list(dataset["train"]["tokens"])
val_tokens = list(dataset["validation"]["tokens"])
test_tokens = list(dataset["test"]["tokens"])

print(f"Sentences -> train: {len(train_tokens)}, val: {len(val_tokens)}, test: {len(test_tokens)}")

token_sequences_raw = train_tokens + val_tokens + test_tokens
token_sequences = remove_punctuation_tokens(token_sequences_raw)

removed_tokens = sum(len(s) for s in token_sequences_raw) - sum(len(s) for s in token_sequences)
print(f"Removed punctuation tokens: {removed_tokens}")

word_to_id, id_to_word, counts = build_vocab(
    token_sequences,
    max_vocab=None,
)
encoded = encode_sentences(token_sequences, word_to_id)
centers, contexts = build_skipgram_pairs(
    encoded,
    window_size=2,
    max_pairs=None,
)

unique_words = len({tok.lower() for sent in token_sequences for tok in sent})
print(f"Unique non-punctuation words in corpus: {unique_words}")
print(f"Vocab size: {len(word_to_id)}")
print(f"Training pairs: {len(centers)}")

neg_probs = build_negative_sampling_distribution(encoded, vocab_size=len(word_to_id), power=0.75)

model = SkipGramNegativeSamplingScratch(vocab_size=len(word_to_id), emb_dim=32, lr=10)

batch_size = 256
epochs = 8
num_negatives = 8
num_samples = len(centers)

for epoch in range(1, epochs + 1):
    order = np.random.permutation(num_samples)
    epoch_loss = 0.0

    for start in range(0, num_samples, batch_size):
        idx = order[start : start + batch_size]
        center_batch = centers[idx]
        context_batch = contexts[idx]
        negative_batch = sample_negative_ids(context_batch, num_negatives=num_negatives, neg_probs=neg_probs)

        loss = model.forward(center_batch, context_batch, negative_batch)
        model.backward()
        model.step()

        epoch_loss += loss * len(idx)

    epoch_loss /= num_samples
    print(f"Epoch {epoch}/{epochs} - negative-sampling loss: {epoch_loss:.4f}")

Sentences -> train: 14041, val: 3250, test: 3453
Removed punctuation tokens: 38376
Unique non-punctuation words in corpus: 26834
Vocab size: 26834
Training pairs: 928906
Epoch 1/8 - negative-sampling loss: 3.6778
Epoch 2/8 - negative-sampling loss: 2.7382
Epoch 3/8 - negative-sampling loss: 2.5195
Epoch 4/8 - negative-sampling loss: 2.3993
Epoch 5/8 - negative-sampling loss: 2.3142
Epoch 6/8 - negative-sampling loss: 2.2485
Epoch 7/8 - negative-sampling loss: 2.1962
Epoch 8/8 - negative-sampling loss: 2.1523


In [6]:
# ------------------------------
# Quick qualitative check
# ------------------------------

def most_similar(query_word, top_k=8):
    if query_word not in word_to_id:
        return []

    W = model.input_embedding.W
    qid = word_to_id[query_word]
    qv = W[qid]

    norms = np.linalg.norm(W, axis=1) + 1e-12
    sims = (W @ qv) / (norms * (np.linalg.norm(qv) + 1e-12))

    best = np.argsort(-sims)[: top_k + 1]
    out = []
    for i in best:
        w = id_to_word[i]
        if w == query_word:
            continue
        out.append((w, float(sims[i])))
        if len(out) == top_k:
            break
    return out

for probe in ["london", "paris", "germany", "man", "woman", "king"]:
    print(f"\nNearest to '{probe}':")
    print(most_similar(probe, top_k=6))


Nearest to 'london':
[('karachi', 0.8380838632583618), ('hong', 0.8101465702056885), ('harare', 0.7990351915359497), ('johannesburg', 0.7887546420097351), ('frankfurt', 0.777804434299469), ('kong', 0.7762638926506042)]

Nearest to 'paris':
[('athens', 0.7948217988014221), ('bangkok', 0.7736629843711853), ('sofia', 0.7572972178459167), ('tirana', 0.7536761164665222), ('nairobi', 0.7522084712982178), ('betar', 0.7496774196624756)]

Nearest to 'germany':
[('italy', 0.7795004844665527), ('denmark', 0.7039278149604797), ('austria', 0.7015149593353271), ('blanc', 0.6865668892860413), ('neumann', 0.6790806651115417), ('9.', 0.6776067614555359)]

Nearest to 'man':
[('right-foot', 0.8437625169754028), ('47-year-old', 0.8184839487075806), ('elephant', 0.8119623064994812), ('woman', 0.7922725081443787), ('30-metre', 0.7913857102394104), ('device', 0.782109797000885)]

Nearest to 'woman':
[('man', 0.7922725081443787), ('ducruet', 0.7908485531806946), ('47-year-old', 0.7792235016822815), ('elephan

In [7]:
# ------------------------------
# Word analogy: a - b + c
# ------------------------------

def analogy(a, b, c, top_k=10):
    a = a.lower()
    b = b.lower()
    c = c.lower()

    missing = [w for w in [a, b, c] if w not in word_to_id]
    if missing:
        print("Missing from vocabulary:", missing)
        return []

    W = model.input_embedding.W
    va = W[word_to_id[a]]
    vb = W[word_to_id[b]]
    vc = W[word_to_id[c]]

    target = va - vb + vc

    norms = np.linalg.norm(W, axis=1) + 1e-12
    target_norm = np.linalg.norm(target) + 1e-12
    sims = (W @ target) / (norms * target_norm)

    for w in [a, b, c]:
        sims[word_to_id[w]] = -1e9

    best_ids = np.argsort(-sims)[:top_k]
    return [(id_to_word[i], float(sims[i])) for i in best_ids]


print("Analogy: king - man + woman")
print(analogy("king", "man", "woman", top_k=10))

print("\nAnalogy: paris - france + germany")
print(analogy("paris", "france", "germany", top_k=10))

print("\nAnalogy: bigger - big + small")
print(analogy("bigger", "big", "small", top_k=10))

Analogy: king - man + woman
[('jacques', 0.7321443557739258), ('coach', 0.6982360482215881), ('galatasaray', 0.6687423586845398), ('peter', 0.6578491926193237), ('body', 0.6555078029632568), ('ronald', 0.6530179381370544), ('nhl', 0.6480898261070251), ('jean', 0.6473462581634521), ('abdou', 0.6447428464889526), ('eldest', 0.6432696580886841)]

Analogy: paris - france + germany
[('boston', 0.6605973243713379), ('ottawa', 0.6510526537895203), ('21', 0.6305766105651855), ('warsaw', 0.6091148257255554), ('veritas', 0.596034049987793), ('herzliya', 0.5904634594917297), ('built', 0.5874097347259521), ('palace', 0.5844117403030396), ('calgary', 0.5840367078781128), ('jersey', 0.5837274193763733)]

Analogy: bigger - big + small
[('tribal', 0.7836201786994934), ('worthing', 0.783565878868103), ('pornographic', 0.7810548543930054), ('brain', 0.7783370614051819), ('demonstration', 0.7473239898681641), ('yogic', 0.7440886497497559), ('state-paid', 0.7392606139183044), ('barentsburg', 0.73839807510

In [8]:
# ------------------------------
# Export learned embeddings + load from external class
# ------------------------------

import json
from pathlib import Path
from exported_embeddings import ExportedEmbeddings


export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)

vectors_path = export_dir / "word_vectors.npy"
word_to_id_path = export_dir / "word_to_id.json"
id_to_word_path = export_dir / "id_to_word.json"

# Save the input embedding matrix learned by SGNS.
np.save(vectors_path, model.input_embedding.W)

with open(word_to_id_path, "w", encoding="utf-8") as f:
    json.dump(word_to_id, f, ensure_ascii=False)

# JSON keys must be strings for portability.
id_to_word_json = {str(k): v for k, v in id_to_word.items()}
with open(id_to_word_path, "w", encoding="utf-8") as f:
    json.dump(id_to_word_json, f, ensure_ascii=False)

print("Saved:")
print(vectors_path)
print(word_to_id_path)
print(id_to_word_path)

emb = ExportedEmbeddings(vectors_path, word_to_id_path, id_to_word_path)
print("\nClass loaded vectors shape:", emb.vectors.shape)
print("Most similar to 'king':", emb.most_similar("king", top_k=5))
print("Analogy king - man + woman:", emb.analogy("king", "man", "woman", top_k=5))

Saved:
exports\word_vectors.npy
exports\word_to_id.json
exports\id_to_word.json

Class loaded vectors shape: (26834, 32)
Most similar to 'king': [('jacques', 0.74261075258255), ('belo', 0.7159354090690613), ('president', 0.688580334186554), ('ronald', 0.6871738433837891), ('facto', 0.6828899383544922)]
Analogy king - man + woman: [('jacques', 0.7321443557739258), ('coach', 0.6982360482215881), ('galatasaray', 0.6687423586845398), ('peter', 0.6578491926193237), ('body', 0.6555078029632568)]
